# Guitar Tab Transcription Demo

Single file inference demo with visualization

In [1]:
import sys
from pathlib import Path
import json
import random
import torch
import numpy as np

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.dadagp_parser import parse_dadagp_file, dadagp_to_events
from src.tab_dataset import (
    build_vocabulary, events_to_ids, event_to_token_string,
    NoteOnEvent, NoteOffEvent, TimeShiftEvent, TabEvent
)
from src.model import FrettingTransformer
from src.metrics import compute_tablature_accuracy
from src.visualization import render_as_tablature, render_as_notes

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

/tmp2/b10401006/.symlinks/miniforge3/envs/guitar-tab/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
import os 
if Path.cwd().name == "notebooks":
    os.chdir(Path(os.getcwd()).parent)

## Configuration

In [3]:
# Paths
DATA_DIR = Path("./DadaGP")
TEST_FILES_JSON = Path("./data_splits/test_files.json")
CHECKPOINT_PATH = Path("./outputs/2025-12-05_03-10/best_model.pt")

# Model config (should match training)
MODEL_CONFIG = {
    'd_model': 128,
    'd_ff': 1024,
    'num_layers': 3,
    'num_heads': 4,
    'dropout_rate': 0.1
}

# Generation config
MAX_LENGTH = 1024
NUM_BEAMS = 1

## Load and Parse a Test File

In [4]:
# Load test files list
with open(TEST_FILES_JSON, 'r') as f:
    test_files = json.load(f)

print(f"Total test files: {len(test_files)}")

# Pick a random test file
random.seed(42)
selected_file = random.choice(test_files)
token_file = selected_file + ".tokens.txt"

print(f"\nSelected file: {Path(selected_file).name}")
print(f"Token file: {token_file}")

Total test files: 519

Selected file: Yoshimatsu, Takashi - Angels In Twilight.gp4
Token file: DadaGP-v1.1/Y/Yoshimatsu, Takashi/Yoshimatsu, Takashi - Angels In Twilight.gp4.tokens.txt


In [5]:
# Parse DadaGP tokens
dadagp_tokens = parse_dadagp_file(token_file)
print(f"Loaded {len(dadagp_tokens)} DadaGP tokens")

# Convert to events
input_events, output_events, bar_positions = dadagp_to_events(dadagp_tokens)

print(f"\nInput events: {len(input_events)} (NOTE_ON, NOTE_OFF, TIME_SHIFT)")
print(f"Output events: {len(output_events)} (NOTE_ON, NOTE_OFF, TIME_SHIFT, TAB)")
print(f"Bar positions: {len(bar_positions)}")

# Show first few events
print("\nFirst 10 input events:")
for i, event in enumerate(input_events[:10]):
    print(f"  {i}: {event_to_token_string(event)}")

print("\nFirst 10 output events:")
for i, event in enumerate(output_events[:10]):
    print(f"  {i}: {event_to_token_string(event)}")

Loaded 521 DadaGP tokens

Input events: 673 (NOTE_ON, NOTE_OFF, TIME_SHIFT)
Output events: 927 (NOTE_ON, NOTE_OFF, TIME_SHIFT, TAB)
Bar positions: 40

First 10 input events:
  0: NOTE_ON_40
  1: NOTE_ON_57
  2: NOTE_OFF_40
  3: NOTE_OFF_57
  4: TIME_SHIFT_240
  5: NOTE_ON_45
  6: NOTE_OFF_45
  7: TIME_SHIFT_240
  8: NOTE_ON_43
  9: NOTE_ON_48

First 10 output events:
  0: NOTE_ON_40
  1: TAB_1_0
  2: NOTE_ON_57
  3: TAB_4_2
  4: NOTE_OFF_40
  5: NOTE_OFF_57
  6: TIME_SHIFT_240
  7: NOTE_ON_45
  8: TAB_2_0
  9: NOTE_OFF_45


## Build Vocabularies and Tokenize

In [6]:
# Build vocabularies
input_vocab, output_vocab = build_vocabulary(
    max_pitch=127,
    max_time_shift=500,
    num_strings=6,
    num_frets=21
)

print(f"Input vocab size: {input_vocab.vocab_size}")
print(f"Output vocab size: {output_vocab.vocab_size}")

# Convert to token IDs
input_ids = events_to_ids(input_events, input_vocab)
output_ids = events_to_ids(output_events, output_vocab)

print(f"\nInput sequence length: {len(input_ids)}")
print(f"Output sequence length: {len(output_ids)}")

Input vocab size: 760
Output vocab size: 886

Input sequence length: 673
Output sequence length: 927


## Truncate to Max Length (if needed)

In [7]:
# Truncate sequences if too long
original_input_len = len(input_ids)
original_output_len = len(output_ids)

if len(input_ids) > MAX_LENGTH:
    print(f"Truncating input from {len(input_ids)} to {MAX_LENGTH}")
    input_ids = input_ids[:MAX_LENGTH]
    input_events = input_events[:MAX_LENGTH]

if len(output_ids) > MAX_LENGTH:
    print(f"Truncating output from {len(output_ids)} to {MAX_LENGTH}")
    output_ids = output_ids[:MAX_LENGTH]
    output_events = output_events[:MAX_LENGTH]

# Convert to tensors
input_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)  # [1, L_in]
target_tensor = torch.tensor(output_ids, dtype=torch.long).unsqueeze(0).to(device)  # [1, L_out]

print(f"\nInput tensor shape: {input_tensor.shape}")
print(f"Target tensor shape: {target_tensor.shape}")


Input tensor shape: torch.Size([1, 673])
Target tensor shape: torch.Size([1, 927])


## Load Model

In [8]:
# Create model
model = FrettingTransformer(
    input_vocab_size=input_vocab.vocab_size,
    output_vocab_size=output_vocab.vocab_size,
    model_config=MODEL_CONFIG
).to(device)

# Load checkpoint
if CHECKPOINT_PATH.exists():
    print(f"Loading checkpoint from {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"  Loaded from epoch {checkpoint['epoch']}")
    print(f"  Val loss: {checkpoint['val_loss']:.4f}")
else:
    print(f"WARNING: Checkpoint not found at {CHECKPOINT_PATH}")
    print("Using untrained model!")

model.eval()
print("\nModel ready for inference")

Created custom T5 model:
  Encoder vocab: 760
  Decoder vocab: 886
  d_model: 128
  d_ff: 1024
  layers: 3
  heads: 4
  parameters: 2,489,216
Loading checkpoint from outputs/2025-12-05_03-10/best_model.pt


  Loaded from epoch 1
  Val loss: 3.9725

Model ready for inference


## Run Prediction

In [9]:
# Generate prediction
with torch.no_grad():
    print("Generating prediction...")
    prediction = model.generate(
        input_ids=input_tensor,
        max_length=MAX_LENGTH,
        num_beams=NUM_BEAMS,
        pad_token_id=output_vocab.pad_id,
        eos_token_id=output_vocab.eos_id
    )

print(f"Generated sequence shape: {prediction.shape}")
print(f"Generated {prediction.shape[1]} tokens")

Generating prediction...
Generated sequence shape: torch.Size([1, 1024])
Generated 1024 tokens


## Pad/Trim to Match Target Length

In [10]:
# Pad or trim prediction to match target length
target_len = target_tensor.shape[1]
pred_len = prediction.shape[1]

if pred_len < target_len:
    # Pad
    padding = torch.full(
        (1, target_len - pred_len),
        output_vocab.pad_id,
        dtype=prediction.dtype,
        device=prediction.device
    )
    prediction = torch.cat([prediction, padding], dim=1)
    print(f"Padded prediction from {pred_len} to {target_len}")
elif pred_len > target_len:
    # Trim
    prediction = prediction[:, :target_len]
    print(f"Trimmed prediction from {pred_len} to {target_len}")
else:
    print(f"Prediction length matches target: {target_len}")

print(f"\nFinal shapes:")
print(f"  Prediction: {prediction.shape}")
print(f"  Target:     {target_tensor.shape}")

Trimmed prediction from 1024 to 927

Final shapes:
  Prediction: torch.Size([1, 927])
  Target:     torch.Size([1, 927])


## Compute Metrics

In [11]:
# Compute accuracy metrics
metrics = compute_tablature_accuracy(
    predictions=prediction,
    targets=target_tensor,
    output_vocab=output_vocab,
    pad_id=output_vocab.pad_id
)

print("="*60)
print("METRICS")
print("="*60)
print(f"Token Accuracy:  {metrics.token_accuracy:.2%}")
print(f"Pitch Accuracy:  {metrics.pitch_accuracy:.2%}")
print(f"Tab Accuracy:    {metrics.tab_accuracy:.2%}")
print(f"Total Tokens:    {metrics.total_tokens:,}")
print(f"Total Notes:     {metrics.total_notes:,}")
print("="*60)

METRICS
Token Accuracy:  7.55%
Pitch Accuracy:  0.00%
Tab Accuracy:    0.00%
Total Tokens:    927
Total Notes:     254


## Visualization - Convert IDs back to Events

In [12]:
def ids_to_events(token_ids, vocab):
    """Convert token IDs back to Event objects."""
    events = []
    
    for token_id in token_ids:
        if token_id == vocab.pad_id:
            break
        
        token_str = vocab.id_to_token.get(token_id, "UNK")
        
        if token_str.startswith("NOTE_ON_"):
            pitch = int(token_str.split("_")[-1])
            events.append(NoteOnEvent(pitch=pitch))
        elif token_str.startswith("NOTE_OFF_"):
            pitch = int(token_str.split("_")[-1])
            events.append(NoteOffEvent(pitch=pitch))
        elif token_str.startswith("TIME_SHIFT_"):
            delta = int(token_str.split("_")[-1])
            events.append(TimeShiftEvent(delta=delta))
        elif token_str.startswith("TAB_"):
            parts = token_str.split("_")
            string = int(parts[1])
            fret = int(parts[2])
            events.append(TabEvent(string=string, fret=fret))
    
    return events

# Convert predictions back to events
pred_events = ids_to_events(prediction[0].cpu().tolist(), output_vocab)
print(f"Converted {len(pred_events)} prediction events")

Converted 926 prediction events


## Input Visualization (MIDI Notes Only)

In [13]:
# Visualize input as note notation
print(render_as_notes(input_events, max_bars=4, chars_per_beat=8, bars_per_row=2))

Note Notation (like tablature, showing note names)
 |A3--A2--C3------A2------D4--G#3-|D3------E3------D3--A2--C3------|
 |             Bar 1              |             Bar 2              |

 |G2------B3--A2--D3------C3------|A3--A2--C3------A2------D4--G#3-|
 |             Bar 3              |             Bar 4              |

... (26 more bars not shown)


## Target Visualization (Ground Truth)

In [14]:
# Target as note notation
print(render_as_notes(output_events, max_bars=4, chars_per_beat=8, bars_per_row=2))

Note Notation (like tablature, showing note names)
 |A3--A2--C3------A2------D4--G#3-|D3------E3------D3--A2--C3------|
 |             Bar 1              |             Bar 2              |

 |G2------B3--A2--D3------C3------|A3--A2--C3------A2------D4--G#3-|
 |             Bar 3              |             Bar 4              |

... (26 more bars not shown)


In [15]:
# Target as guitar tablature
print(render_as_tablature(output_events, max_bars=4, chars_per_beat=8, bars_per_row=2))

Guitar Tablature (Standard Tuning: EADGBE)
e|0-------3-------5-------7-------|--------12------7-------0-------|
B|----0---3-----------------------|5-------------------0---3-------|
G|----------------------------6---|----------------0---------------|
D|2-----------------------7-------|--------------------------------|
A|--------------------------------|--------------------------------|
E|--------------------------------|--------------------------------|
 |             Bar 1              |             Bar 2              |

e|3-------0-----------------------|0-------3-------5-------7-------|
B|------------0---5-------3-------|----0---3-----------------------|
G|--------------------------------|----------------------------6---|
D|--------4-----------------------|2-----------------------7-------|
A|--------------------------------|--------------------------------|
E|--------------------------------|--------------------------------|
 |             Bar 3              |             Bar 4      

## Prediction Visualization

In [16]:
# Prediction as note notation
print(render_as_notes(pred_events, max_bars=4, chars_per_beat=8, bars_per_row=2))

Note Notation (like tablature, showing note names)
 |--------------------------------|
 |             Bar 1              |


In [17]:
# Prediction as guitar tablature
print(render_as_tablature(pred_events, max_bars=4, chars_per_beat=8, bars_per_row=2))

Guitar Tablature (Standard Tuning: EADGBE)
e|--------------------------------|--------------------------------|
B|--------------------------------|--------------------------------|
G|--------------------------------|--------------------------------|
D|--------------------------------|--------------------------------|
A|--------------------------------|--------------------------------|
E|--------------------------------|--------------------------------|
 |             Bar 1              |             Bar 2              |

e|--------------------------------|--------------------------------|
B|--------------------------------|--------------------------------|
G|--------------------------------|--------------------------------|
D|--------------------------------|--------------------------------|
A|--------------------------------|--------------------------------|
E|--------------------------------|--------------------------------|
 |             Bar 3              |             Bar 4      

## Side-by-Side Comparison

In [18]:
pred_events


[TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEvent(type='TIME_SHIFT', delta=240),
 TimeShiftEve

In [19]:
# Compare first N notes
def extract_note_tab_pairs(events, max_notes=15):
    """Extract (pitch, string, fret) tuples from events."""
    notes = []
    i = 0
    
    while i < len(events) and len(notes) < max_notes:
        if isinstance(events[i], NoteOnEvent):
            pitch = events[i].pitch
            # Check if next event is TAB
            if i + 1 < len(events) and isinstance(events[i + 1], TabEvent):
                string = events[i + 1].string
                fret = events[i + 1].fret
                notes.append((pitch, string, fret))
                i += 2
            else:
                notes.append((pitch, None, None))
                i += 1
        else:
            i += 1
    
    return notes

target_notes = extract_note_tab_pairs(output_events, max_notes=15)
pred_notes = extract_note_tab_pairs(pred_events, max_notes=15)

print("="*90)
print("FIRST 15 NOTES COMPARISON")
print("="*90)
print(f"{'#':<4} {'TARGET':<35} {'PREDICTION':<35} {'MATCH'}")
print("-"*90)

for i in range(max(len(target_notes), len(pred_notes))):
    if i < len(target_notes):
        p, s, f = target_notes[i]
        target_str = f"pitch={p:3d} → str={s} fret={f:2d}" if s is not None else f"pitch={p:3d}"
    else:
        target_str = "---"
    
    if i < len(pred_notes):
        p, s, f = pred_notes[i]
        pred_str = f"pitch={p:3d} → str={s} fret={f:2d}" if s is not None else f"pitch={p:3d}"
    else:
        pred_str = "---"
    
    match = "✓" if target_str == pred_str else "✗"
    print(f"{i:<4} {target_str:<35} {pred_str:<35} {match}")

print("="*90)

FIRST 15 NOTES COMPARISON
#    TARGET                              PREDICTION                          MATCH
------------------------------------------------------------------------------------------
0    pitch= 40 → str=1 fret= 0           ---                                 ✗
1    pitch= 57 → str=4 fret= 2           ---                                 ✗
2    pitch= 45 → str=2 fret= 0           ---                                 ✗
3    pitch= 43 → str=1 fret= 3           ---                                 ✗
4    pitch= 48 → str=2 fret= 3           ---                                 ✗
5    pitch= 45 → str=1 fret= 5           ---                                 ✗
6    pitch= 47 → str=1 fret= 7           ---                                 ✗
7    pitch= 62 → str=4 fret= 7           ---                                 ✗
8    pitch= 56 → str=3 fret= 6           ---                                 ✗
9    pitch= 50 → str=2 fret= 5           ---                                 ✗
10   pitch

## Error Analysis

In [20]:
# Analyze types of errors
def analyze_errors(target_notes, pred_notes):
    """Analyze prediction errors."""
    pitch_errors = []
    tab_errors = []
    correct_notes = []
    
    for i in range(min(len(target_notes), len(pred_notes))):
        t_pitch, t_str, t_fret = target_notes[i]
        p_pitch, p_str, p_fret = pred_notes[i]
        
        if t_pitch == p_pitch:
            if t_str == p_str and t_fret == p_fret:
                correct_notes.append((t_pitch, t_str, t_fret))
            else:
                tab_errors.append((t_pitch, (t_str, t_fret), (p_str, p_fret)))
        else:
            pitch_errors.append((t_pitch, p_pitch))
    
    return pitch_errors, tab_errors, correct_notes

pitch_errors, tab_errors, correct_notes = analyze_errors(target_notes, pred_notes)

print("="*60)
print("ERROR ANALYSIS")
print("="*60)
print(f"Correct notes:   {len(correct_notes)}")
print(f"Pitch errors:    {len(pitch_errors)}")
print(f"Tab errors:      {len(tab_errors)} (pitch correct, but wrong string/fret)")
print()

if pitch_errors:
    print("Sample pitch errors (target → prediction):")
    for target_p, pred_p in pitch_errors[:5]:
        print(f"  pitch {target_p} → {pred_p}")

if tab_errors:
    print("\nSample tab errors (correct pitch, wrong position):")
    for pitch, (t_s, t_f), (p_s, p_f) in tab_errors[:5]:
        print(f"  pitch {pitch}: str={t_s} fret={t_f:2d} → str={p_s} fret={p_f:2d}")

print("="*60)

ERROR ANALYSIS
Correct notes:   0
Pitch errors:    0
Tab errors:      0 (pitch correct, but wrong string/fret)

